# Simulation Globale LMTL

Ce notebook configure et exécute une simulation globale du modèle LMTL en utilisant les forçages issus du fichier Zarr fourni.
Pour l'instant, le transport est désactivé.


In [ ]:
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pint
import xarray as xr
from IPython.display import Markdown

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController
from seapopym.lmtl.configuration import LMTLParams
from seapopym.lmtl.core import (
    compute_day_length,
    compute_gillooly_temperature,
    compute_layer_weighted_mean,
    compute_mortality_tendency,
    compute_production_dynamics,
    compute_production_initialization,
    compute_recruitment_age,
    compute_threshold_temperature,
)
from seapopym.standard.coordinates import Coordinates

In [ ]:
ureg = pint.get_application_registry()

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

## 1. Chargement et Préparation des Données


In [ ]:
# Chemin vers le fichier Zarr
zarr_path = "/Users/adm-lehodey/Documents/Workspace/Projects/phd_optimization/notebooks/Article_1/data/1_global/post_processed_light_global_multiyear_bgc_001_033.zarr"

print(f"Chargement des données depuis : {zarr_path}")
ds_raw = xr.open_zarr(zarr_path)

# Renommage des dimensions pour correspondre au standard Seapopym
# T -> time, Z -> z, Y -> y, X -> x
ds = ds_raw.rename(
    {
        "T": Coordinates.T.value,
        "Z": Coordinates.Z.value,
        "Y": Coordinates.Y.value,
        "X": Coordinates.X.value,
    }
)
ds.x.attrs = {}
ds.y.attrs = {}
# Affichage pour vérification
ds

## 2. Configuration de la Simulation


In [ ]:
# Définition de la période de simulation
# On prend une courte période pour tester (ex: 1 mois)
start_date = "1998-01-02"
end_date = "2019-12-31"
timestep = timedelta(days=1)  # Pas de temps journalier pour commencer

config = SimulationConfig(
    start_date=start_date,
    end_date=end_date,
    timestep=timestep,
)

# Paramètres LMTL spécifiés
lmtl_params = LMTLParams(
    day_layer=0,
    night_layer=0,
    tau_r_0=10.38 * ureg.days,
    gamma_tau_r=ureg.Quantity(0.11, ureg.degC**-1),
    lambda_0=ureg.Quantity(1 / 150, ureg.day**-1),
    gamma_lambda=ureg.Quantity(0.15, ureg.degC**-1),
    E=0.1668,
    T_ref=ureg.Quantity(0, ureg.degC),
)

print("Configuration :")
print(f"  Période : {start_date} -> {end_date}")
print(f"  Pas de temps : {timestep}")
print(f"  Paramètres LMTL : {lmtl_params}")

## 3. Préparation des Forçages pour la Simulation


In [ ]:
# Sélection temporelle pour alléger le chargement si nécessaire
# Note : SimulationController gère l'interpolation, mais on peut réduire le dataset en entrée
forcings = ds.sel({Coordinates.T.value: slice("1998-01-01", "2020-01-01")})
forcings = forcings[["primary_production", "temperature"]].load()

# Vérification des variables requises
# Le blueprint attend 'temperature' et 'primary_production'
# Elles sont présentes dans le dataset.

# Création des cohortes
cohorts = (np.arange(0, np.ceil(lmtl_params.tau_r_0.magnitude) + 1) * ureg.day).to("second")
cohorts_da = xr.DataArray(
    cohorts.magnitude, dims=["cohort"], name="cohort", attrs={"units": "second"}
)

# Ajout des paramètres statiques aux forçages (pour qu'ils soient accessibles)
forcings = forcings.assign_coords(cohort=cohorts_da)
forcings["dt"] = config.timestep.total_seconds()

# IMPORTANT: Normaliser les unités pour que Pint puisse les parser
# Remplacer "mg m-2 day-1" par "mg/m**2/day"
if "primary_production" in forcings:
    forcings["primary_production"].attrs["units"] = "mg/m**2/day"

# Remplacer "degree_Celsius" si nécessaire (normalement OK)
if "temperature" in forcings and forcings["temperature"].attrs.get("units") in ["degC", "deg_C"]:
    forcings["temperature"].attrs["units"] = "degree_Celsius"

# CONVERSION MANUELLE: Le système de conversion automatique ne fonctionne pas
# Convertir NPP de mg/m²/day vers g/m²/s
print("=== Conversion du NPP ===")
print(
    f"Avant: NPP range = {forcings['primary_production'].min().values:.2e} - {forcings['primary_production'].max().values:.2e} mg/m²/day"
)

# forcings["primary_production"] = forcings["primary_production"] / 1000.0 / 86400.0
# forcings["primary_production"].attrs["units"] = "g/m**2/second"

print(
    f"Après: NPP range = {forcings['primary_production'].min().values:.2e} - {forcings['primary_production'].max().values:.2e} g/m²/s"
)

forcings

## 4. État Initial


In [ ]:
# Création de l'état initial (vide pour commencer)
# Dimensions spatiales
lats = forcings[Coordinates.Y.value]
lons = forcings[Coordinates.X.value]

# Biomasse initiale nulle
biomass_init = xr.DataArray(
    np.zeros((len(lats), len(lons))),
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons},
    dims=(Coordinates.Y.value, Coordinates.X.value),
    name="Zooplankton/biomass",
)
biomass_init.attrs = {"units": "g/m**2"}

# Production initiale nulle
production_init = xr.DataArray(
    np.zeros((len(lats), len(lons), len(cohorts))),
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons, "cohort": cohorts_da},
    dims=(Coordinates.Y.value, Coordinates.X.value, "cohort"),
    name="Zooplankton/production",
)
production_init.attrs = {"units": "g/m**2/day"}

initial_state = xr.Dataset(
    {"Zooplankton/biomass": biomass_init, "Zooplankton/production": production_init}
)
initial_state

In [ ]:
def configure_model(bp):
    # Enregistrement des forçages
    # Note: temperature est 4D (time, z, y, x) dans le dataset
    bp.register_forcing(
        "temperature",
        dims=(Coordinates.T.value, Coordinates.Z.value, Coordinates.Y.value, Coordinates.X.value),
        units="degree_Celsius",
    )
    bp.register_forcing(
        "primary_production",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="g/m**2/second",
    )

    bp.register_forcing("dt")
    bp.register_forcing("cohort")
    bp.register_forcing(Coordinates.T.value)
    bp.register_forcing(Coordinates.Y.value)

    bp.register_group(
        group_prefix="Zooplankton",
        units=[
            {
                "func": compute_day_length,
                "output_mapping": {"output": "day_length"},
                "input_mapping": {"latitude": Coordinates.Y.value, "time": Coordinates.T.value},
                "output_units": {"output": "dimensionless"},
            },
            {
                "func": compute_layer_weighted_mean,
                "input_mapping": {"forcing": "temperature"},
                "output_mapping": {"output": "mean_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_threshold_temperature,
                "input_mapping": {"temperature": "mean_temperature", "min_temperature": "T_ref"},
                "output_mapping": {"output": "thresholded_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_gillooly_temperature,
                "input_mapping": {"temperature": "thresholded_temperature"},
                "output_mapping": {"output": "gillooly_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_recruitment_age,
                "input_mapping": {"temperature": "gillooly_temperature"},
                "output_mapping": {"output": "recruitment_age"},
                "output_units": {"output": "second"},
            },
            {
                "func": compute_production_initialization,
                "input_mapping": {"cohorts": "cohort"},
                "output_mapping": {"output": "production_source_npp"},
                "output_tendencies": {"output": "production"},
                "output_units": {"output": "g/m**2/second"},
            },
            {
                "func": compute_production_dynamics,
                "input_mapping": {
                    "cohort_ages": "cohort",
                    "dt": "dt",
                },
                "output_mapping": {
                    "production_tendency": "production_tendency",
                    "recruitment_source": "biomass_source",
                },
                "output_tendencies": {
                    "production_tendency": "production",
                    "recruitment_source": "biomass",
                },
                "output_units": {
                    "production_tendency": "g/m**2/second",
                    "recruitment_source": "g/m**2/second",
                },
            },
            {
                "func": compute_mortality_tendency,
                "input_mapping": {"temperature": "gillooly_temperature"},
                "output_mapping": {"mortality_loss": "biomass_mortality"},
                "output_tendencies": {"mortality_loss": "biomass"},
                "output_units": {"mortality_loss": "g/m**2/second"},
            },
        ],
        parameters={
            "day_layer": {"units": "dimensionless"},
            "night_layer": {"units": "dimensionless"},
            "tau_r_0": {"units": "second"},
            "gamma_tau_r": {"units": "1/degree_Celsius"},
            "lambda_0": {"units": "1/second"},
            "gamma_lambda": {"units": "1/degree_Celsius"},
            "T_ref": {"units": "degree_Celsius"},
            "E": {"units": "dimensionless"},
        },
        state_variables={
            "production": {
                "dims": (Coordinates.Y.value, Coordinates.X.value, "cohort"),
                "units": "g/m**2/second",
            },
            "biomass": {
                "dims": (Coordinates.Y.value, Coordinates.X.value),
                "units": "g/m**2",
            },
        },
    )


bp = Blueprint()
configure_model(bp)
Markdown("```mermaid\n" + bp.export_mermaid() + "\n```")

## 6. Exécution


In [ ]:
controller = SimulationController(config)
controller.setup(
    configure_model,
    initial_state,
    forcings,
    parameters={"Zooplankton": lmtl_params},
    output_variables=["Zooplankton/biomass"],
)

print("Démarrage de la simulation...")
controller.run()
print("Simulation terminée.")

## 7. Visualisation


In [ ]:
results = controller.results


plt.figure(figsize=(12, 6))
results["Zooplankton/biomass"].mean("time").plot()
plt.title(f"Biomasse Zooplankton à la fin de la simulation ({end_date})")
plt.show()

In [ ]:
results["Zooplankton/biomass"].rename("biomass").to_zarr(
    "./zooplankton_no_transport_seapopym_v2.zarr", mode="w"
)

In [ ]:
controller.state